In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parents[0]
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
METRICS_DIR = ROOT / "reports" / "metrics"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROCESSED_DIR / "features.parquet")
df["date"] = pd.to_datetime(df["date"])

print(df.shape)
df.head(2)


(57473650, 29)


,id,item_id,dept_id,cat_id,store_id,state_id,d,y,date,wm_yr_wk,...,snap_TX,snap_WI,sell_price,lag_1,lag_7,lag_28,roll_mean_7,roll_std_7,roll_mean_28,roll_std_28
0,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_29,2,2011-02-26,11105,...,0,0,2.0,4.0,1.0,3.0,13.714286,9.446591,10.535714,6.203238
1,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_30,2,2011-02-27,11105,...,0,0,2.0,2.0,2.0,0.0,0.000000,0.000000,0.000000,0.000000


In [2]:
# Target
y = df["y"].astype(np.float32)

# Drop leakage / identifiers
drop_cols = {"y", "date", "d", "wm_yr_wk"}  # keep item/store ids as features
X = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Make sure these are categorical if present
cat_cols = ["item_id","dept_id","cat_id","store_id","state_id",
            "event_name_1","event_type_1","event_name_2","event_type_2"]

for c in cat_cols:
    if c in X.columns:
        X[c] = X[c].astype("category")

# We'll keep `id` out to avoid overfitting to unique series id
if "id" in X.columns:
    X = X.drop(columns=["id"])

print("X shape:", X.shape)


X shape: (57473650, 24)


In [3]:
max_date = df["date"].max()
valid_start = max_date - pd.Timedelta(days=27)

train_idx = df["date"] < valid_start
valid_idx = df["date"] >= valid_start

X_train, y_train = X.loc[train_idx], y.loc[train_idx]
X_valid, y_valid = X.loc[valid_idx], y.loc[valid_idx]

print("Train:", X_train.shape, "Valid:", X_valid.shape)
print("Valid range:", valid_start.date(), "→", max_date.date())


Train: (56619930, 24) Valid: (853720, 24)
Valid range: 2016-03-28 → 2016-04-24


In [4]:
import lightgbm as lgb
from joblib import dump
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as num

def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.clip(np.abs(y_true), 1e-8, None)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

model = lgb.LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=128,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="l1",
    callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=True)]
)

pred = model.predict(X_valid)

metrics = {
    "mae": float(mean_absolute_error(y_valid, pred)),
    "rmse": float(np.sqrt(mean_squared_error(y_valid, pred))),
    "mape": mape(y_valid, pred),
    "best_iteration": int(getattr(model, "best_iteration_", model.n_estimators)),
    "n_train_rows": int(X_train.shape[0]),
    "n_valid_rows": int(X_valid.shape[0]),
    "n_features": int(X_train.shape[1]),
    "valid_start": str(valid_start.date()),
    "valid_end": str(max_date.date())
}
metrics

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.244426 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4654
[LightGBM] [Info] Number of data points in the train set: 56619930, number of used features: 24
[LightGBM] [Info] Start training from score 1.126407
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2988]	valid_0's l1: 0.990589	valid_0's l2: 3.7098


{'mae': 0.9905893869168887,
 'rmse': 1.9260834199880488,
 'mape': 3254574937.2810235,
 'best_iteration': 2988,
 'n_train_rows': 56619930,
 'n_valid_rows': 853720,
 'n_features': 24,
 'valid_start': '2016-03-28',
 'valid_end': '2016-04-24'}